In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import pandas as pd
import numpy as np
from pyMyriad import *
from pyMyriad.tabular import flatten, tabulate
from pyMyriad.listing import gt_table, simple_table

In [21]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "Education": np.random.normal(0, 1, 1000),
  "Employed": np.random.choice([0, 1], 1000)
})

In [22]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
efun = lambda df: np.mean(df.Education)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .summarize_by(m = mfun, n = nfun)\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun, label = "Income analysis")\
  .analyze_by(m = efun, label = "Education analysis")

res = atree.run(df)


# Tabulation with simple_table()

The `simple_table()` function provides a lightweight way to display analysis results as a pandas DataFrame without requiring the great-tables package.

In [23]:
# Basic simple_table - shows only analysis results
simple_table(res)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49374.017389
3,,,--,--,--,--,n,14858.297631
8,,,Benin Y/N,False,Age Group,age40,m,50604.634556
8,,,,,,,n,13808.336687
9,,,,,,,m,-0.038035
11,,,,,,age60,m,48679.548216
11,,,,,,,n,14750.914045
12,,,,,,,m,-0.141607
16,,,,True,,age40,m,50802.637366
16,,,,,,,n,15150.245505


## simple_table with pivot

You can pivot by a split variable to show results across columns:

In [24]:
# Pivot by Gender to show results side-by-side
simple_table(res, by="df.Gender")

pivot_lvl,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,F,M
0,--,--,--,--,m,49374.017389,49553.216202
1,--,--,--,--,n,14858.297631,14304.783613
2,Benin Y/N,False,Age Group,age40,m,-0.038035,-0.045641
3,,,,,m,50604.634556,50470.392551
4,,,,,n,13808.336687,13248.767677
5,,,,age60,m,-0.141607,0.068372
6,,,,,m,48679.548216,47027.065573
7,,,,,n,14750.914045,13293.803723
8,,True,,age40,m,0.032339,0.120846
9,,,,,m,50802.637366,50064.964293


## simple_table with all nodes

Include non-analysis nodes (splits and summaries) by setting `include_non_analysis=True`:

In [25]:
# Show all nodes including splits
simple_table(res, include_non_analysis=True).head(10)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
0,--,--,--,--,--,--,None,None
1,Gender,--,--,--,--,--,None,None
2,,F,--,--,--,--,None,None
3,,,--,--,--,--,m,49374.017389
3,,,--,--,--,--,n,14858.297631
4,,,Benin Y/N,--,--,--,None,None
5,,,,False,--,--,None,None
6,,,,,Age Group,--,None,None
7,,,,,,age40,None,None
8,,,,,,,m,50604.634556


## simple_table without duplicate suppression

By default, duplicate values in hierarchy columns are suppressed for cleaner display. You can disable this:

In [26]:
# Show all duplicate values
simple_table(res, suppress_duplicates=False).head()

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49374.017389
3,Gender,F,--,--,--,--,n,14858.297631
8,Gender,F,Benin Y/N,False,Age Group,age40,m,50604.634556
8,Gender,F,Benin Y/N,False,Age Group,age40,n,13808.336687
9,Gender,F,Benin Y/N,False,Age Group,age40,m,-0.038035


# Tabulation with gt_table()

The `gt_table()` function creates beautifully formatted tables using the great-tables package. It provides more styling options and is ideal for reports and presentations.

In [27]:
# Basic gt_table with title
gt_table(res, title="Analysis Results").show()

Analysis Results 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 m 
 49374.01738876354 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14858.29763072662 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 50604.6345558043 
 
 
 
 
 
 
 
 
 n 
 13808.336686567627 
 
 
 
 
 
 
 
 
 m 
 -0.03803487285905186 
 
 
 
 
 
 
 
 age60 
 m 
 48679.548215688876 
 
 
 
 
 
 
 
 
 n 
 14750.914044916639 
 
 
 
 
 
 
 
 
 m 
 -0.141606822845729 
 
 
 
 
 
 True 
 
 age40 
 m 
 50802.63736589126 
 
 
 
 
 
 
 
 
 n 
 15150.24550522379 
 
 
 
 
 
 
 
 
 m 
 0.03233936175486183 
 
 
 
 
 
 
 
 age60 
 m 
 51582.47606231179 
 
 
 
 
 
 
 
 
 n 
 14846.837225695106 
 
 
 
 
 
 
 
 
 m 
 0.05951567637478195 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 m 
 49553.21620160617 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14304.783612948055 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 50470.3925507557 
 
 
 
 
 
 
 
 
 n 
 13248.7676770048 
 
 
 
 
 
 
 
 
 m 
 -0.04564056220627714 
 
 
 
 
 
 
 
 age60 
 m 
 47027.06557301037 
 
 
 
 
 
 
 
 
 n 
 13293.803722838882 
 
 
 
 
 
 
 
 
 m 
 0.06837150381353414 
 
 
 
 
 
 True 
 
 age40 
 m 
 50064.96429333218 
 
 
 
 
 
 
 
 
 n 
 14993.778758579865 
 
 
 
 
 
 
 
 
 m 
 0.12084626565120102 
 
 
 
 
 
 
 
 age60 
 m 
 50068.92602750659 
 
 
 
 
 
 
 
 
 n 
 15993.661698284246 
 
 
 
 
 
 
 
 
 m 
 0.0734618133488765

## gt_table with pivot

Pivot by a split variable:

In [28]:
# Pivot by Gender with custom title and subtitle
gt_table(
    res, 
    by="df.Gender", 
    title="Gender Comparison",
    subtitle="Income and Education Analysis by Gender"
).show()

Gender Comparison 
 
 
 Income and Education Analysis by Gender 
 
 
 
 Hierarchy 
 
 Statistic 
 F 
 M 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49374.01738876354 
 49553.21620160617 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14858.29763072662 
 14304.783612948055 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 -0.03803487285905186 
 -0.04564056220627714 
 
 
 
 
 
 
 m 
 50604.6345558043 
 50470.3925507557 
 
 
 
 
 
 
 n 
 13808.336686567627 
 13248.7676770048 
 
 
 
 
 
 age60 
 m 
 -0.141606822845729 
 0.06837150381353414 
 
 
 
 
 
 
 m 
 48679.548215688876 
 47027.06557301037 
 
 
 
 
 
 
 n 
 14750.914044916639 
 13293.803722838882 
 
 
 
 True 
 
 age40 
 m 
 0.03233936175486183 
 0.12084626565120102 
 
 
 
 
 
 
 m 
 50802.63736589126 
 50064.96429333218 
 
 
 
 
 
 
 n 
 15150.24550522379 
 14993.778758579865 
 
 
 
 
 
 age60 
 m 
 0.05951567637478195 
 0.0734618133488765 
 
 
 
 
 
 
 m 
 51582.47606231179 
 50068.92602750659 
 
 
 
 
 
 
 n 
 14846.837225695106 
 15993.661698284246

## gt_table with all nodes and custom formatting

Include non-analysis nodes and customize decimal places:

In [29]:
# Show all nodes with 2 decimal places
gt_table(
    res, 
    title="Complete Analysis Tree",
    include_non_analysis=True,
    decimals=2
).show()

Complete Analysis Tree 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 Gender 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 F 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49374.01738876354 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14858.29763072662 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50604.6345558043 
 
 
 
 
 
 
 
 
 n 
 13808.336686567627 
 
 
 
 
 
 
 
 
 m 
 -0.03803487285905186 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 48679.548215688876 
 
 
 
 
 
 
 
 
 n 
 14750.914044916639 
 
 
 
 
 
 
 
 
 m 
 -0.141606822845729 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50802.63736589126 
 
 
 
 
 
 
 
 
 n 
 15150.24550522379 
 
 
 
 
 
 
 
 
 m 
 0.03233936175486183 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 51582.47606231179 
 
 
 
 
 
 
 
 
 n 
 14846.837225695106 
 
 
 
 
 
 
 
 
 m 
 0.05951567637478195 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49553.21620160617 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14304.783612948055 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50470.3925507557 
 
 
 
 
 
 
 
 
 n 
 13248.7676770048 
 
 
 
 
 
 
 
 
 m 
 -0.04564056220627714 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 47027.06557301037 
 
 
 
 
 
 
 
 
 n 
 13293.803722838882 
 
 
 
 
 
 
 
 
 m 
 0.06837150381353414 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50064.96429333218 
 
 
 
 
 
 
 
 
 n 
 14993.778758579865 
 
 
 
 
 
 
 
 
 m 
 0.12084626565120102 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50068.92602750659 
 
 
 
 
 
 
 
 
 n 
 15993.661698284246 
 
 
 
 
 
 
 
 
 m 
 0.0734618133488765

# Pivoting Statistics as Columns

The `pivot_statistics` parameter allows you to display statistics as columns instead of rows, creating a wider, more compact table format.

In [30]:
# Pivot statistics into columns - simple_table
simple_table(res, pivot_statistics=True)

statistics,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,m,n
0,Gender,F,--,--,--,--,49374.017389,14858.297631
1,,M,--,--,--,--,49553.216202,14304.783613
2,,F,Benin Y/N,False,Age Group,age40,-0.038035,NaN
3,,,,,,,50604.634556,13808.336687
4,,,,,,age60,-0.141607,NaN
5,,,,,,,48679.548216,14750.914045
6,,,,True,,age40,0.032339,NaN
7,,,,,,,50802.637366,15150.245505
8,,,,,,age60,0.059516,NaN
9,,,,,,,51582.476062,14846.837226


## Combining pivot_statistics with by

You can combine both pivots to create a matrix-style table with split groups and statistics as columns:

In [31]:
# Combine both pivots - shows Gender groups with statistics as columns
simple_table(res, by="df.Gender", pivot_statistics=True)

,_Level_0,_Level_1,_Level_2,_Level_3,F||m,F||n,M||m,M||n
0,--,--,--,--,49374.017389,14858.297631,49553.216202,14304.783613
1,Benin Y/N,False,Age Group,age40,-0.038035,NaN,-0.045641,NaN
2,,,,,50604.634556,13808.336687,50470.392551,13248.767677
3,,,,age60,-0.141607,NaN,0.068372,NaN
4,,,,,48679.548216,14750.914045,47027.065573,13293.803723
5,,True,,age40,0.032339,NaN,0.120846,NaN
6,,,,,50802.637366,15150.245505,50064.964293,14993.778759
7,,,,age60,0.059516,NaN,0.073462,NaN
8,,,,,51582.476062,14846.837226,50068.926028,15993.661698


## gt_table with pivot_statistics

The same feature works with `gt_table()` for beautifully formatted output:

In [32]:
# gt_table with statistics as columns
gt_table(
    res,
    pivot_statistics=True,
    title="Analysis with Statistics as Columns"
).show()

Analysis with Statistics as Columns 
 
 
 
 Hierarchy 
 
 m 
 n 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 49374.01738876354 
 14858.29763072662 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 49553.21620160617 
 14304.783612948055 
 
 
 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.03803487285905186 
 
 
 
 
 
 
 
 
 
 50604.6345558043 
 13808.336686567627 
 
 
 
 
 
 
 
 age60 
 -0.141606822845729 
 
 
 
 
 
 
 
 
 
 48679.548215688876 
 14750.914044916639 
 
 
 
 
 
 True 
 
 age40 
 0.03233936175486183 
 
 
 
 
 
 
 
 
 
 50802.63736589126 
 15150.24550522379 
 
 
 
 
 
 
 
 age60 
 0.05951567637478195 
 
 
 
 
 
 
 
 
 
 51582.47606231179 
 14846.837225695106 
 
 
 
 M 
 
 False 
 
 age40 
 -0.04564056220627714 
 
 
 
 
 
 
 
 
 
 50470.3925507557 
 13248.7676770048 
 
 
 
 
 
 
 
 age60 
 0.06837150381353414 
 
 
 
 
 
 
 
 
 
 47027.06557301037 
 13293.803722838882 
 
 
 
 
 
 True 
 
 age40 
 0.12084626565120102 
 
 
 
 
 
 
 
 
 
 50064.96429333218 
 14993.778758579865 
 
 
 
 
 
 
 
 age60 
 0.0734618133488765 
 
 
 
 
 
 
 
 
 
 50068.92602750659 
 15993.661698284246

In [33]:
# gt_table combining both pivots
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Gender Comparison - Matrix View",
    subtitle="Each group-statistic combination as a column"
).show()

Gender Comparison - Matrix View 
 
 
 Each group-statistic combination as a column 
 
 
 
 Hierarchy 
 
 
 F 
 
 
 M 
 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 m 
 n 
 m 
 n 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 49374.01738876354 
 14858.29763072662 
 49553.21620160617 
 14304.783612948055 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.03803487285905186 
 
 -0.04564056220627714 
 
 
 
 
 
 
 
 50604.6345558043 
 13808.336686567627 
 50470.3925507557 
 13248.7676770048 
 
 
 
 
 
 age60 
 -0.141606822845729 
 
 0.06837150381353414 
 
 
 
 
 
 
 
 48679.548215688876 
 14750.914044916639 
 47027.06557301037 
 13293.803722838882 
 
 
 
 True 
 
 age40 
 0.03233936175486183 
 
 0.12084626565120102 
 
 
 
 
 
 
 
 50802.63736589126 
 15150.24550522379 
 50064.96429333218 
 14993.778758579865 
 
 
 
 
 
 age60 
 0.05951567637478195 
 
 0.0734618133488765 
 
 
 
 
 
 
 
 51582.47606231179 
 14846.837225695106 
 50068.92602750659 
 15993.661698284246

In [34]:
print(res)

Data Tree
  Split: df.Gender
    └- F
        analysis: 
        └- m: 49374.01738876354
        └- n: 14858.29763072662
      Split: Benin Y/N
        └- False
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 50604.6345558043
                └- n: 13808.336686567627
                analysis: Education analysis
                └- m: -0.03803487285905186
            └- age60
                analysis: Income analysis
                └- m: 48679.548215688876
                └- n: 14750.914044916639
                analysis: Education analysis
                └- m: -0.141606822845729
        └- True
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 50802.63736589126
                └- n: 15150.24550522379
                analysis: Education analysis
                └- m: 0.03233936175486183
            └- age60
                analysis: Income analysis
              

In [44]:
from pyMyriad.tabular import format_statistics
rr = format_statistics(
    res, 
    label="Income analysis",  # Only format nodes with this label
    m="{m:.0f} /// {n:.0f}",         # Create statistic 'm' with this format
    inplace=False, 
    remove_original=True
)

gt_table(rr, title="Formatted Statistics").show()

Formatted Statistics 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 m 
 49374.01738876354 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14858.29763072662 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 50605 /// 13808 
 
 
 
 
 
 
 
 
 m 
 -0.03803487285905186 
 
 
 
 
 
 
 
 age60 
 m 
 48680 /// 14751 
 
 
 
 
 
 
 
 
 m 
 -0.141606822845729 
 
 
 
 
 
 True 
 
 age40 
 m 
 50803 /// 15150 
 
 
 
 
 
 
 
 
 m 
 0.03233936175486183 
 
 
 
 
 
 
 
 age60 
 m 
 51582 /// 14847 
 
 
 
 
 
 
 
 
 m 
 0.05951567637478195 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 m 
 49553.21620160617 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14304.783612948055 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 50470 /// 13249 
 
 
 
 
 
 
 
 
 m 
 -0.04564056220627714 
 
 
 
 
 
 
 
 age60 
 m 
 47027 /// 13294 
 
 
 
 
 
 
 
 
 m 
 0.06837150381353414 
 
 
 
 
 
 True 
 
 age40 
 m 
 50065 /// 14994 
 
 
 
 
 
 
 
 
 m 
 0.12084626565120102 
 
 
 
 
 
 
 
 age60 
 m 
 50069 /// 15994 
 
 
 
 
 
 
 
 
 m 
 0.0734618133488765